# Titanic Survival Prediction: An Iterative Model-Building Study

**Author:** Avdhesh Thakur
**Competition:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)
**Metric:** Accuracy (fraction of test passengers correctly classified)

---

## Purpose of this notebook

This is not a single model. It is a **documented study of five successive modelling approaches**, each one a deliberate experiment to test a hypothesis about what would improve generalisation on the Titanic dataset.

The value of this notebook is the **reasoning and honest measurement** behind each step, not the final leaderboard number. For every version I record the cross-validation (CV) accuracy and the public leaderboard score, and I discuss *why* a change helped or did not.

### Summary of results

| Version | Approach | CV accuracy | Public LB | Outcome |
|--------:|----------|:-----------:|:---------:|---------|
| v1 | Random Forest, basic features | 0.8227 | 0.77272 | Baseline |
| v2 | Gradient Boosting + extra features (FarePerPerson, Deck) | 0.8317 | 0.76076 | CV up, LB **down** (overfit) |
| v3 | Tuned Random Forest, trimmed clean features | 0.8305 | **0.77990** | Best LB |
| v4 | v3 + survival-grouped titles | 0.8271 | 0.77751 | No gain |
| v5 | Soft-voting ensemble of 5 models | **0.8372** | 0.77990 | Best CV, tied best LB |

**Headline lesson:** a higher CV score did not reliably translate to a higher leaderboard score. The simplest clean model (v3) and a five-model ensemble (v5) both plateaued at 0.77990, which is close to the practical ceiling for honest models on this dataset.

### How CV is measured here

All CV scores use **5-fold cross-validation** on the 891-row training set (`cross_val_score(..., cv=5)`). This gives an honest estimate of out-of-sample accuracy before touching the leaderboard. The public leaderboard is itself computed on only a subset of the 418-row test set, which is why small CV differences do not separate cleanly on the board.


## 0. Setup and shared data loading

On Kaggle the competition files are mounted under `/kaggle/input/competitions/titanic/`. If you run this notebook locally, point `DATA_DIR` at the folder containing `train.csv` and `test.csv`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

import os
# Kaggle path first, then a couple of common local fallbacks.
CANDIDATES = [
    "/kaggle/input/competitions/titanic",
    "/kaggle/input/titanic",
    "../data",
    "data",
]
DATA_DIR = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, "train.csv"))), CANDIDATES[0])
print("Using data directory:", DATA_DIR)

def load_raw():
    train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
    return train, test

train_raw, test_raw = load_raw()
print("train:", train_raw.shape, "| test:", test_raw.shape)

---

## Version 1 - Baseline Random Forest

**Hypothesis:** a standard Random Forest on a handful of sensible features is a reasonable starting point.

**Features:** Title (extracted from name, rare titles grouped), FamilySize, IsAlone, Age (filled by title-group median), Fare, Embarked, Sex. The Cabin column is dropped (mostly missing).

**Model:** `RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3)`.

**Result:** CV 0.8227, leaderboard 0.77272. A solid baseline.

In [ ]:
def fe_v1(train, test):
    train = train.copy(); test = test.copy()
    train["is_train"] = 1; test["is_train"] = 0; test["Survived"] = np.nan
    full = pd.concat([train, test], ignore_index=True)
    full["Title"] = full["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
    rare = (full["Title"].value_counts() < 10)
    full["Title"] = full["Title"].apply(lambda t: "Rare" if rare.get(t, False) else t)
    full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
    full["IsAlone"] = (full["FamilySize"] == 1).astype(int)
    full["Age"] = full.groupby("Title")["Age"].transform(lambda s: s.fillna(s.median()))
    full["Age"] = full["Age"].fillna(full["Age"].median())
    full["Fare"] = full["Fare"].fillna(full["Fare"].median())
    full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
    full["Sex"] = full["Sex"].map({"male": 0, "female": 1})
    full = pd.get_dummies(full, columns=["Embarked", "Title"], drop_first=True)
    return full

def split_xy(full):
    drop = ["PassengerId", "Survived", "Name", "Ticket", "Cabin", "is_train"]
    feats = [c for c in full.columns if c not in drop]
    tr = full[full["is_train"] == 1]; te = full[full["is_train"] == 0]
    return tr[feats], tr["Survived"].astype(int), te[feats], te, feats

full_v1 = fe_v1(train_raw, test_raw)
X1, y1, Xt1, te1, feats1 = split_xy(full_v1)
m1 = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3, random_state=42)
print("v1 CV accuracy: %.4f" % cross_val_score(m1, X1, y1, cv=5).mean())

---

## Version 2 - Gradient Boosting with extra features

**Hypothesis:** a more powerful model (Gradient Boosting) plus richer features (fare per person, cabin deck) will improve accuracy.

**New features:** `FarePerPerson = Fare / FamilySize` and `Deck` (first letter of Cabin, U for unknown).

**Model:** `GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05)`.

**Result:** CV **rose** to 0.8317, but the leaderboard **fell** to 0.76076. This is the key cautionary result of the study: the added power and the sparse Deck feature let the model fit the training set more closely without generalising. Classic mild overfitting.

In [ ]:
def fe_v2(train, test):
    train = train.copy(); test = test.copy()
    train["is_train"] = 1; test["is_train"] = 0; test["Survived"] = np.nan
    full = pd.concat([train, test], ignore_index=True)
    full["Title"] = full["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
    rare = (full["Title"].value_counts() < 10)
    full["Title"] = full["Title"].apply(lambda t: "Rare" if rare.get(t, False) else t)
    full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
    full["IsAlone"] = (full["FamilySize"] == 1).astype(int)
    full["Fare"] = full["Fare"].fillna(full["Fare"].median())
    full["FarePerPerson"] = full["Fare"] / full["FamilySize"]            # NEW
    full["Deck"] = full["Cabin"].astype(str).str[0]                      # NEW
    full.loc[~full["Deck"].isin(list("ABCDEFGT")), "Deck"] = "U"
    full["Age"] = full.groupby("Title")["Age"].transform(lambda s: s.fillna(s.median()))
    full["Age"] = full["Age"].fillna(full["Age"].median())
    full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
    full["Sex"] = full["Sex"].map({"male": 0, "female": 1})
    full = pd.get_dummies(full, columns=["Embarked", "Title", "Deck"], drop_first=True)
    return full

full_v2 = fe_v2(train_raw, test_raw)
X2, y2, Xt2, te2, feats2 = split_xy(full_v2)
m2 = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
print("v2 CV accuracy: %.4f" % cross_val_score(m2, X2, y2, cv=5).mean())

---

## Version 3 - Trimmed features, tuned Random Forest (best leaderboard)

**Hypothesis:** after v2 overfit, *removing* the noisy feature and using a shallow, well-regularised model will generalise better.

**Change:** drop the sparse Deck feature entirely; keep only features with a clear link to survival. Use a shallow Random Forest (`max_depth=5`, `min_samples_leaf=4`).

**Result:** CV 0.8305, leaderboard **0.77990 - the best of the study.** Confirms the hypothesis: on a small, noisy dataset, a cleaner and simpler model wins.

In [ ]:
full_v3 = fe_v1(train_raw, test_raw)  # same clean feature set as v1 (no Deck, no FarePerPerson)
X3, y3, Xt3, te3, feats3 = split_xy(full_v3)
m3 = RandomForestClassifier(n_estimators=400, max_depth=5, min_samples_leaf=4, random_state=42)
print("v3 CV accuracy: %.4f" % cross_val_score(m3, X3, y3, cv=5).mean())

---

## Version 4 - Survival-grouped titles

**Hypothesis:** grouping titles by their survival pattern (Master, Mr, Mrs, Miss, Officer, Royalty) gives the model cleaner signal than lumping rare titles into one bucket.

**Change:** a hand-built title map replacing the "rare titles together" rule from v1/v3.

**Result:** CV 0.8271, leaderboard 0.77751. Slightly *below* v3. The richer title categories did not help on the test set here; the extra granularity added variance without a matching gain.

In [ ]:
def fe_v4(train, test):
    train = train.copy(); test = test.copy()
    train["is_train"] = 1; test["is_train"] = 0; test["Survived"] = np.nan
    full = pd.concat([train, test], ignore_index=True)
    full["Title"] = full["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
    title_map = {
        "Mr": "Mr", "Mrs": "Mrs", "Miss": "Miss", "Master": "Master",
        "Mme": "Mrs", "Ms": "Miss", "Mlle": "Miss",
        "Dr": "Officer", "Rev": "Officer", "Col": "Officer", "Major": "Officer", "Capt": "Officer",
        "Lady": "Royalty", "Sir": "Royalty", "Don": "Royalty", "Dona": "Royalty",
        "Countess": "Royalty", "Jonkheer": "Royalty",
    }
    full["Title"] = full["Title"].map(title_map).fillna("Rare")
    full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
    full["IsAlone"] = (full["FamilySize"] == 1).astype(int)
    full["Fare"] = full["Fare"].fillna(full["Fare"].median())
    full["Age"] = full.groupby("Title")["Age"].transform(lambda s: s.fillna(s.median()))
    full["Age"] = full["Age"].fillna(full["Age"].median())
    full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
    full["Sex"] = full["Sex"].map({"male": 0, "female": 1})
    full = pd.get_dummies(full, columns=["Embarked", "Title"], drop_first=True)
    return full

full_v4 = fe_v4(train_raw, test_raw)
X4, y4, Xt4, te4, feats4 = split_xy(full_v4)
m4 = RandomForestClassifier(n_estimators=400, max_depth=5, min_samples_leaf=4, random_state=42)
print("v4 CV accuracy: %.4f" % cross_val_score(m4, X4, y4, cv=5).mean())

---

## Version 5 - Soft-voting ensemble (best CV)

**Hypothesis:** combining several diverse models that make different errors will cancel out individual mistakes and generalise best.

**Models combined (soft voting on predicted probabilities):** Random Forest, Extra Trees, Gradient Boosting, Logistic Regression (scaled), and an SVM (scaled, probability-calibrated).

**Result:** CV **0.8372 - the highest of the study** - but the leaderboard landed at 0.77990, tied with v3. The ensemble is a genuinely stronger, more robust model, yet the public score plateaued, evidence that ~0.78 is near the practical ceiling for honest models on this data.

In [ ]:
full_v5 = fe_v4(train_raw, test_raw)  # grouped-title features from v4
X5, y5, Xt5, te5, feats5 = split_xy(full_v5)

rf  = RandomForestClassifier(n_estimators=400, max_depth=5, min_samples_leaf=4, random_state=42)
et  = ExtraTreesClassifier(n_estimators=400, max_depth=6, min_samples_leaf=4, random_state=42)
gb  = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42)
lr  = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=0.5))
svc = make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(C=1.0, random_state=42), ensemble=False))

m5 = VotingClassifier(
    estimators=[("rf", rf), ("et", et), ("gb", gb), ("lr", lr), ("svc", svc)],
    voting="soft",
)
print("v5 CV accuracy: %.4f" % cross_val_score(m5, X5, y5, cv=5).mean())

---

## Producing the submission file

The best leaderboard score came from **v3** (and v5 tied it). Below we fit v3 on all training data and write `submission.csv`. Swap in `m5`/`X5`/`Xt5` to reproduce the ensemble submission instead.

In [ ]:
best_model, Xb, yb, Xtb, teb = m3, X3, y3, Xt3, te3

best_model.fit(Xb, yb)
preds = best_model.predict(Xtb).astype(int)
submission = pd.DataFrame({
    "PassengerId": teb["PassengerId"].astype(int),
    "Survived": preds,
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv:", submission.shape)
submission.head()

---

## Closing notes

Five deliberate experiments, each testing a clear hypothesis and measured the same honest way:

1. **v1** set a baseline.
2. **v2** showed that more power + more features can *reduce* generalisation (overfitting).
3. **v3** recovered and improved by simplifying, the best leaderboard result.
4. **v4** showed a reasonable-sounding feature idea that did not pay off.
5. **v5** built the strongest, most robust model (best CV) but hit the data's practical ceiling.

The recurring theme: **trust honest out-of-sample measurement over intuition about model complexity**, and remember that on a small dataset the leaderboard has limited resolving power.